# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "is_married",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

# Readind data from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Some Data Transformations


#Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Normalization

In [0]:
df = df.filter(F.col("cst_id").isNotNull())

In [0]:
df = (
    df.withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status"))== "S", 0)
        .when(F.upper(F.col("cst_marital_status"))== "M", 1)
        .otherwise(None)
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr"))== "F", "Female")
        .when(F.upper(F.col("cst_gndr"))== "M", "Male")
        .otherwise("n/a")
    )  
)

In [0]:
df.count()

In [0]:
window = Window.partitionBy("cst_id").orderBy(F.col("cst_create_date").desc())

df = (
    df.withColumn("rn", F.row_number().over(window))
      .filter(F.col("rn") == 1)
      .drop("rn")
)

In [0]:
df.count()

#Renaming the Columns

In [0]:
for old_name , new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# Write Into Silver Table

In [0]:
(
    df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("silver.crm_customers")   
)

In [0]:
df.count()